# 🍽️ ChefMind RAG System
**Hybrid Retrieval-Augmented Generation for Recipe Q&A**

| Step | Component |
|------|----------|
| 1 | Install packages |
| 2 | Load or build indexes (smart: Drive cache → build from datasets) |
| 3 | Retrieval functions |
| 4 | Cross-Encoder reranker |
| 5 | Load ChefMind LoRA model |
| 5b | Name lookup from 3 datasets |
| 6 | RAG pipeline |
| 7 | Expose via ngrok API |

## 📦 Step 1: Install Packages

In [1]:
%%capture
!pip install datasets langchain langchain-community langchain-core \
             langchain-text-splitters langchain-huggingface \
             faiss-cpu rank-bm25 \
             sentence-transformers pyngrok nest-asyncio fastapi uvicorn \
             transformers peft accelerate bitsandbytes -q
print("✅ Packages installed!")

## ♻️ Step 2: Load or Build Indexes

**Smart logic:**
- ✅ Indexes found on Google Drive → load in ~30 seconds
- 🔨 Not found → build from 267k recipes (~15 min), then save to Drive automatically

Either way, you end up with `vectorstore`, `bm25`, and `chunks` ready to use.

In [2]:
import os, pickle
from google.colab import drive
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from rank_bm25 import BM25Okapi

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/ChefMind/indexes"

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

faiss_path = os.path.join(DRIVE_DIR, "faiss_index")
bm25_path  = os.path.join(DRIVE_DIR, "bm25.pkl")
chunk_path = os.path.join(DRIVE_DIR, "chunks.pkl")

indexes_exist = (
    os.path.isdir(faiss_path) and
    os.path.isfile(bm25_path) and
    os.path.isfile(chunk_path)
)

if indexes_exist:
    # ── Fast path: load from Drive ────────────────────────────────────────
    print("✅ Indexes found on Drive — loading...")
    vectorstore = FAISS.load_local(
        faiss_path, embeddings, allow_dangerous_deserialization=True
    )
    with open(bm25_path, "rb") as f:
        bm25 = pickle.load(f)
    with open(chunk_path, "rb") as f:
        chunks = pickle.load(f)
    print(f"✅ Loaded {len(chunks):,} chunks from Drive")

else:
    # ── First-run path: build from datasets, then save to Drive ──────────
    print("⚠️  Indexes not found — building from scratch (~15 min first time)...")
    from datasets import load_dataset

    # ── Per-dataset formatters ─────────────────────────────────────────────
    # Each formatter returns ONE complete recipe string.
    # We keep each recipe as a single Document (no chunking) so the model
    # always receives a full, coherent recipe as RAG context — never a
    # mid-recipe fragment.

    def fmt_simple(row):
        """
        tengomucho/simple_recipes  &  formido/recipes
        Columns: 'names' (str), 'recipes' (str — cleaned bullet steps)
        No separate ingredients column in these datasets.
        """
        name  = str(row.get('names', '')).strip().title()
        steps = str(row.get('recipes', '')).strip()
        return (
            f"Recipe: {name}\n"
            f"Instructions:\n{steps}"
        )

    def fmt_rahul(row):
        """
        rahul7star/food-recipes
        Columns: 'name', 'ingredients' (list), 'steps' (list),
                 'minutes', 'description'
        Full structured dataset — ingredients and steps are separate lists.
        """
        name  = str(row.get('name', '')).strip().title()
        desc  = str(row.get('description', '')).strip()
        mins  = row.get('minutes', '')

        raw_ing = row.get('ingredients', [])
        ing_block = (
            '\n'.join(f"- {i}" for i in raw_ing if str(i).strip())
            if isinstance(raw_ing, list) else str(raw_ing).strip()
        )

        raw_steps = row.get('steps', [])
        step_block = (
            '\n'.join(f"{n+1}. {s}" for n, s in enumerate(raw_steps) if str(s).strip())
            if isinstance(raw_steps, list) else str(raw_steps).strip()
        )

        return (
            f"Recipe: {name}\n"
            + (f"Description: {desc}\n" if desc else "")
            + (f"Cook time: {mins} min\n" if mins else "")
            + f"Ingredients:\n{ing_block}\n"
            + f"Instructions:\n{step_block}"
        )

    documents = []

    print("⏳ Loading ds1: tengomucho/simple_recipes...")
    ds1 = load_dataset("tengomucho/simple_recipes", split="train")
    documents += [fmt_simple(r) for r in ds1]
    print(f"   ✅ {len(ds1):,} recipes")

    print("⏳ Loading ds2: rahul7star/food-recipes...")
    ds2 = load_dataset("rahul7star/food-recipes", split="train")
    documents += [fmt_rahul(r) for r in ds2]
    print(f"   ✅ {len(ds2):,} recipes")

    print("⏳ Loading ds3: formido/recipes...")
    ds3 = load_dataset("formido/recipes", split="train")
    documents += [fmt_simple(r) for r in ds3]
    print(f"   ✅ {len(ds3):,} recipes")

    print(f"\n📚 Total documents: {len(documents):,}")

    # ── One Document per recipe (no mid-recipe chunking) ──────────────────
    # Each recipe stays whole so RAG context is always a complete recipe,
    # never a fragment. Recipes are naturally short enough (~300-400 tokens)
    # to fit within the embedding model's 512-token limit.
    chunks = [Document(page_content=doc) for doc in documents if doc.strip()]
    print(f"✅ {len(chunks):,} recipe documents (1 per recipe, no chunking)")

    # ── FAISS ─────────────────────────────────────────────────────────────
    print("⏳ Building FAISS index...")
    vectorstore = FAISS.from_documents(chunks, embeddings)
    print("✅ FAISS ready!")

    # ── BM25 ──────────────────────────────────────────────────────────────
    print("⏳ Building BM25 index...")
    corpus = [c.page_content.lower().split() for c in chunks]
    bm25 = BM25Okapi(corpus)
    print("✅ BM25 ready!")

    # ── Save to Drive ─────────────────────────────────────────────────────
    print("⏳ Saving indexes to Drive (next run will be fast)...")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    vectorstore.save_local(faiss_path)
    with open(bm25_path, "wb") as f:
        pickle.dump(bm25, f)
    with open(chunk_path, "wb") as f:
        pickle.dump(chunks, f)
    print(f"✅ Saved to {DRIVE_DIR} — next run loads in ~30 seconds!")

print("\n🎉 Indexes ready!")


/tmp/ipykernel_32432/3616263452.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Indexes found on Drive — loading...
✅ Loaded 267,335 chunks from Drive

🎉 Indexes ready!


## 🔀 Step 3: Retrieval Functions

In [3]:
def generate_queries(q):
    return [q, f"recipe for {q}", f"how to cook {q}", f"ingredients for {q}"]

def bm25_retrieve(query, k=5):
    scores = bm25.get_scores(query.lower().split())
    top_idx = scores.argsort()[-k:][::-1]
    return [(chunks[i], scores[i]) for i in top_idx]

def rrf_fusion(faiss_docs, bm25_results, k=60):
    scores = {}
    for rank, doc in enumerate(faiss_docs):
        key = doc.page_content[:100]
        scores[key] = scores.get(key, 0) + 1/(k + rank + 1)
    for rank, (doc, _) in enumerate(bm25_results):
        key = doc.page_content[:100]
        scores[key] = scores.get(key, 0) + 1/(k + rank + 1)
    all_docs = {d.page_content[:100]: d for d in faiss_docs}
    all_docs.update({d.page_content[:100]: d for d, _ in bm25_results})
    return [all_docs[k] for k, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True) if k in all_docs]

def hybrid_retrieve(question, k=5):
    return rrf_fusion(
        vectorstore.similarity_search(question, k=k),
        bm25_retrieve(question, k=k)
    )

print("✅ Retrieval functions ready!")

✅ Retrieval functions ready!


## 🤖 Step 4: Cross-Encoder Reranker

In [4]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, docs, top_k=5):
    if not docs:
        return docs
    pairs = [[query, doc.page_content] for doc in docs]
    scores = cross_encoder.predict(pairs)
    return [doc for _, doc in sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)[:top_k]]

print("✅ Cross-Encoder ready!")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ Cross-Encoder ready!


## 🧠 Step 5: Load ChefMind LoRA Model
*Takes ~3 minutes*

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct"
LORA_REPO  = "Osamaelbably87/chefmind-lora"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("⏳ Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(LORA_REPO)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"⏳ Loading base model {BASE_MODEL} in 4-bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"⏳ Applying LoRA adapter from {LORA_REPO}...")
model = PeftModel.from_pretrained(base_model, LORA_REPO)
model.eval()
print("✅ Model loaded!")

⏳ Loading tokenizer...
⏳ Loading base model unsloth/Meta-Llama-3.1-8B-Instruct in 4-bit...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

⏳ Applying LoRA adapter from Osamaelbably87/chefmind-lora...
✅ Model loaded!


## 🔤 Step 5b: Recipe Name Lookup
Builds a name index from all 3 datasets so every recipe gets a proper name.

In [6]:
import re, os, json
from datasets import load_dataset

NAMES_PATH = "/content/drive/MyDrive/ChefMind/indexes/recipe_names.json"

if os.path.isfile(NAMES_PATH):
    print("✅ recipe_names.json found on Drive — loading...")
    with open(NAMES_PATH, "r") as f:
        all_names = json.load(f)
    print(f"✅ Loaded {len(all_names):,} recipe names!")

else:
    print("⚠️  Names not cached — building (one-time only)...")
    all_names = []

    try:
        ds1 = load_dataset("tengomucho/simple_recipes", split="train")
        all_names += [r['names'] for r in ds1 if r.get('names','').strip()]
        print(f"✅ ds1: {len(all_names):,} names")
    except Exception as e:
        print(f"⚠️ ds1 failed: {e}")

    try:
        ds2 = load_dataset("rahul7star/food-recipes", split="train")
        cols = ds2.column_names
        name_col = next((c for c in ['title','name','names','recipe_name'] if c in cols), cols[0])
        before = len(all_names)
        all_names += [r[name_col] for r in ds2 if str(r.get(name_col,'')).strip()]
        print(f"✅ ds2: {len(all_names)-before:,} names")
    except Exception as e:
        print(f"⚠️ ds2 failed: {e}")

    try:
        ds3 = load_dataset("formido/recipes", split="train")
        cols = ds3.column_names
        name_col = next((c for c in ['title','name','names','recipe_name'] if c in cols), cols[0])
        before = len(all_names)
        all_names += [r[name_col] for r in ds3 if str(r.get(name_col,'')).strip()]
        print(f"✅ ds3: {len(all_names)-before:,} names")
    except Exception as e:
        print(f"⚠️ ds3 failed: {e}")

    os.makedirs(os.path.dirname(NAMES_PATH), exist_ok=True)
    with open(NAMES_PATH, "w") as f:
        json.dump(all_names, f, ensure_ascii=False)
    print(f"✅ Saved {len(all_names):,} names → next run loads instantly!")

print(f"\n✅ Total: {len(all_names):,} recipe names")


# ── Fixed name lookup ──────────────────────────────────────────────────────
COOKING_STOPWORDS = {
    'and','or','with','the','a','an','in','of','for','to','my','i',
    'baked','fried','grilled','roasted','steamed','stuffed','cooked',
    'made','style','easy','simple','quick','homemade','classic','fresh',
}

def find_best_name(query):
    q_words = set(re.findall(r'\b\w{4,}\b', query.lower())) - COOKING_STOPWORDS
    best_name, best_score = None, 0

    for name in all_names:
        if not name or not str(name).strip():
            continue
        n_words = set(re.findall(r'\b\w{4,}\b', str(name).lower())) - COOKING_STOPWORDS
        score   = len(q_words & n_words)
        if score > best_score:
            best_score = score
            best_name  = str(name).strip().title()

    min_score = 2 if len(q_words) >= 2 else 1
    if best_name and best_score >= min_score:
        return best_name

    parts = [w.strip().title() for w in query.split(',')[:3] if len(w.strip()) > 2]
    return ', '.join(parts) + ' Recipe' if parts else query.title()

print("✅ Name lookup ready!")

✅ recipe_names.json found on Drive — loading...
✅ Loaded 267,335 recipe names!

✅ Total: 267,335 recipe names
✅ Name lookup ready!


## 🍳 Step 6: RAG Pipeline

In [7]:
import json, re, torch

# ── Is this a real cooking step? ──────────────────────────────────────────
def is_real_step(text):
    if len(text.strip()) < 15:
        return False
    if re.match(
        r'^[\d½¼¾⅓⅔\s/]+\s*(cup|tbsp|tsp|oz|lb|g|ml|kg|clove|piece|slice|bunch|pinch)s?\b',
        text.strip(), re.IGNORECASE
    ):
        return False
    action_words = [
        'heat','add','mix','stir','cook','bake','fry','boil','simmer','pour',
        'place','combine','season','remove','serve','chop','cut','preheat',
        'whisk','blend','drain','rinse','marinate','grill','roast','sauté',
        'transfer','cover','let','bring','reduce','set','prepare','coat',
        'sprinkle','spread','flip','turn','rest','allow','brush','toss',
        'squeeze','mince','slice','dice','peel','wash','pat','dry','beat',
        'fold','knead','roll','shape','press','arrange','layer','top','garnish'
    ]
    return any(w in text.lower() for w in action_words)


# ── Is this an actual ingredient (not a leaked cooking step)? ─────────────
def is_ingredient_line(text):
    t = text.strip()
    if not t or len(t) < 2:
        return False
    if len(t) > 80:
        return False
    words = t.split()
    if len(words) > 7:
        return False
    action_re = re.compile(
        r'\b(heat|add|mix|stir|cook|bake|fry|boil|simmer|pour|place|combine|'
        r'season|remove|serve|chop|cut|preheat|whisk|blend|drain|rinse|'
        r'marinate|grill|roast|transfer|cover|bring|reduce|prepare|coat|'
        r'sprinkle|spread|flip|toss|squeeze|mince|dice|peel|wash|pat|beat|'
        r'fold|knead|roll|shape|press|arrange|garnish|until|minutes|degrees|'
        r'simmering|stirring|boiling|adding|cooking|using|bringing|reducing)\b',
        re.IGNORECASE
    )
    if action_re.search(t) and len(words) > 3:
        return False
    return True


# ── Extract ingredients from LLM output ───────────────────────────────────
def extract_ingredients(raw_text, fallback_query):
    # Split on INSTRUCTIONS: — everything before it is the ingredient block
    parts = re.split(r'\n\s*INSTRUCTIONS?\s*:\s*\n', raw_text, maxsplit=1, flags=re.IGNORECASE)
    ing_block = parts[0]
    ing_block = re.sub(r'^INGREDIENTS?\s*:\s*\n', '', ing_block, flags=re.IGNORECASE).strip()

    qty_pattern = re.compile(
        r'^(?P<qty>[\d½¼¾⅓⅔\s/\-]+(?:cup|tbsp|tsp|oz|lb|g|kg|ml|l|clove|piece|slice|bunch|handful|pinch|medium|large|small|can|jar|package|head|stalk|sprig)s?(?:\s+of)?)?\s*(?P<name>.+)',
        re.IGNORECASE
    )
    result = []
    seen = set()
    for line in ing_block.split('\n'):
        line = line.strip()
        if not line:
            continue
        clean = re.sub(r'^[-•*\d.)\s]\s*', '', line).strip()
        if not clean or len(clean) < 2:
            continue
        if not is_ingredient_line(clean):
            continue
        m = qty_pattern.match(clean)
        if m and m.group('qty') and m.group('qty').strip():
            qty  = m.group('qty').strip()
            name = m.group('name').strip().rstrip('.,;')
        else:
            qty  = ''
            name = clean.rstrip('.,;')
        key = name.lower()
        if name and len(name) > 1 and key not in seen:
            seen.add(key)
            result.append({'name': name, 'quantity': qty})
        if len(result) >= 12:
            break

    if not result:
        result = [
            {'name': w.strip(), 'quantity': ''}
            for w in fallback_query.split(',')
            if len(w.strip()) > 2
        ]
    return result


# ── Dynamic difficulty ─────────────────────────────────────────────────────
def calculate_difficulty(instructions, cooking_time_str):
    n_steps = len(instructions)
    mins = 30
    m = re.search(r'(\d+)', cooking_time_str or '')
    if m:
        mins = int(m.group(1))
        if re.search(r'hour', cooking_time_str, re.IGNORECASE):
            mins *= 60
    if n_steps <= 4 and mins <= 20:
        return 'Easy'
    elif n_steps >= 10 or mins >= 60:
        return 'Hard'
    else:
        return 'Medium'

print("✅ Helper functions ready!")

✅ Helper functions ready!


In [8]:
import torch, re

def run_chefmind_pipeline(query):

    # ── 1. Retrieve + rerank ───────────────────────────────────────────────
    retrieved_docs = hybrid_retrieve(query, k=10)
    reranked_docs  = rerank(query, retrieved_docs, top_k=3)

    # ── 2. Build RAG context ───────────────────────────────────────────────
    # FIX: was missing — reranked_docs was computed but never put in the prompt
    context_parts = []
    for doc in reranked_docs[:2]:
        text = doc.page_content.strip()
        if len(text) > 50:
            context_parts.append(text[:400])
    rag_context = '\n---\n'.join(context_parts)

    # ── 3. Prompt ──────────────────────────────────────────────────────────
    system_msg = (
        'You are ChefMind, a recipe assistant. '
        'Output EXACTLY ONE recipe and then STOP. Never write a second recipe.\n\n'
        'Use this strict format:\n\n'
        'INGREDIENTS:\n'
        '- [quantity] [ingredient]\n'
        '(max 8 ingredients)\n\n'
        'INSTRUCTIONS:\n'
        '1. [short cooking action — max 20 words]\n'
        '(max 8 steps)\n\n'
        'CRITICAL RULES:\n'
        '- Only use ingredients that belong in the requested dish.\n'
        '- Instructions must be real cooking actions (heat, add, stir, bake…).\n'
        '- STOP immediately after the last numbered step. No extra text.\n'
    )

    user_msg = ''
    if rag_context:
        user_msg += f'Reference context from recipe database:\n{rag_context}\n\n'
    user_msg += f'Create a recipe for: {query}'

    prompt = (
        '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n'
        f'{system_msg}'
        '<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n'
        f'{user_msg}\n'
        '<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'
        'INGREDIENTS:\n'
    )

    # ── 4. Stop tokens ─────────────────────────────────────────────────────
    # FIX: original had no eos_token_id override → model generated recipe 2+
    stop_ids = [tokenizer.eos_token_id]
    for tok_str in ['<|eot_id|>', '<|end_of_text|>', '<|start_header_id|>']:
        try:
            tid = tokenizer.convert_tokens_to_ids(tok_str)
            if tid and tid != tokenizer.unk_token_id and tid not in stop_ids:
                stop_ids.append(tid)
        except Exception:
            pass

    # ── 5. Generate ────────────────────────────────────────────────────────
    inputs = tokenizer(
        prompt, return_tensors='pt',
        truncation=True, max_length=2048
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=350,
            temperature=0.65,
            do_sample=True,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=stop_ids,          # ← key fix
        )

    raw = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # ── 6. Truncate bleed: repeated punctuation before 2nd recipe ──────────
    # e.g. "Serve hot .,,,,,Chicken Breast Baked In A Bag"
    junk_split = re.split(r'[.,;]{3,}', raw, maxsplit=1)
    if len(junk_split) > 1 and len(junk_split[0]) > 80:
        raw = junk_split[0].strip()

    # ── 7. Truncate bleed: standalone title line before INGREDIENTS header ─
    raw = re.sub(
        r'\n[A-Z][A-Za-z\s,]{10,80}\n(?=\s*(?:INGREDIENTS|INSTRUCTIONS))',
        '\n', raw, flags=re.IGNORECASE
    )

    print('=== RAW OUTPUT (first 600 chars) ===')
    print(raw[:600])
    print('======================================')

    # ── 8. Split into ingredient / instruction blocks ──────────────────────
    parts         = re.split(r'\n\s*INSTRUCTIONS?\s*:\s*\n', raw, maxsplit=1, flags=re.IGNORECASE)
    ing_section   = parts[0] if len(parts) >= 1 else ''
    instr_section = parts[1] if len(parts) >= 2 else raw

    # ── 9. Truncate instruction section at any new-recipe header ──────────
    for stop_pat in [
        r'\n[A-Z][A-Za-z\s,]{5,100}\n\s*(?:INGREDIENTS|Ingredients)',
        r'\n\s*(?:INGREDIENTS|Ingredients)\s*:',
        r'\n\s*Recipe\s*\d+',
        r'\n-{3,}',
    ]:
        sm = re.search(stop_pat, instr_section)
        if sm:
            instr_section = instr_section[:sm.start()]

    # ── 10. Parse ingredients ──────────────────────────────────────────────
    ing_list = extract_ingredients(ing_section, query)

    # ── 11. Parse instructions ─────────────────────────────────────────────
    instr_lines = []
    for line in instr_section.split('\n'):
        line  = line.strip()
        if not line: continue
        clean = re.sub(r'^(?:Step\s*)?\d+[.):\s]\s*', '', line, flags=re.IGNORECASE).strip()
        clean = re.sub(r'^[-•*]\s*', '', clean).strip()

        # Truncate mid-step bleed at repeated punctuation
        junk_at = clean.find('.,,')
        if junk_at == -1: junk_at = clean.find(',,,')
        if junk_at > 8:
            clean = clean[:junk_at].strip().rstrip('.,;!')

        if clean and is_real_step(clean) and len(clean) < 300:
            instr_lines.append(clean)

    if not instr_lines:
        for line in instr_section.split('\n'):
            line  = line.strip()
            if not line: continue
            clean = re.sub(r'^(?:Step\s*)?\d+[.):\s]\s*', '', line).strip()
            clean = re.sub(r'^[-•*]\s*', '', clean).strip()
            if clean and len(clean) > 10:
                instr_lines.append(clean)

    instructions = [f'Step {i+1}: {l}' for i, l in enumerate(instr_lines[:8])]
    if not instructions:
        instructions = ['Prepare all ingredients and cook until done.']

    # ── 12. Cook time + difficulty ─────────────────────────────────────────
    time_match = re.search(
        r'(\d[\d\s\-–]*(?:to\s*\d+)?\s*(?:min(?:ute)?s?|hours?))',
        raw, re.IGNORECASE
    )
    cooking_time = time_match.group(1).strip() if time_match else '30 minutes'
    difficulty   = calculate_difficulty(instructions, cooking_time)

    return {
        'recipe_name':  find_best_name(query),
        'ingredients':  ing_list,
        'instructions': instructions,
        'cooking_time': cooking_time,
        'difficulty':   difficulty,
    }

print('✅ ChefMind Pipeline v2 ready!')

✅ ChefMind Pipeline v2 ready!


## 🌐 Step 7: Expose via ngrok API

In [9]:
from pyngrok import ngrok

NGROK_TOKEN = "3I3ol8qaAYQG4Es4TxoaXwUCJKL_54Ru6FkyNgXgLGPmBmC25"
ngrok.set_auth_token(NGROK_TOKEN)
print("✅ NGROK authenticated!")

✅ NGROK authenticated!


In [10]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import nest_asyncio, uvicorn, threading
from pyngrok import ngrok

nest_asyncio.apply()

app = FastAPI(title="ChefMind API", version="3.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class RecipeRequest(BaseModel):
    query: str

@app.get("/")
def root():
    return {"message": "ChefMind API v3 is running! 🍳"}

@app.get("/health")
def health():
    return {"status": "ok", "model": "ChefMind LoRA v3"}

@app.post("/recipe")
def get_recipe(request: RecipeRequest):
    result = run_chefmind_pipeline(request.query)
    return {
        "recipe_name":  result.get("recipe_name", ""),
        "ingredients":  result.get("ingredients", []),
        "instructions": result.get("instructions", []),
        "cooking_time": result.get("cooking_time", ""),
        "difficulty":   result.get("difficulty", "Medium"),
    }

ngrok_tunnel = ngrok.connect(8000)
print(f"🌐 API live at:  {ngrok_tunnel.public_url}")
print(f"❤️  Health:       {ngrok_tunnel.public_url}/health")
print(f"🍳 Recipe POST:  {ngrok_tunnel.public_url}/recipe")

thread = threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000))
thread.daemon = True
thread.start()
print("✅ Server running!")

🌐 API live at:  https://arguable-clarify-capillary.ngrok-free.dev
❤️  Health:       https://arguable-clarify-capillary.ngrok-free.dev/health
🍳 Recipe POST:  https://arguable-clarify-capillary.ngrok-free.dev/recipe
✅ Server running!
